In [53]:
library(rsample)     # data splitting 
library(dplyr)       # data wrangling
library(rpart)       # performing regression trees
library(janitor)
library(readr)
library(tidyverse)
library(xgboost)
library(rPref)
library(caret)

### Constants

In [54]:
def_outbreak <- 2        # number of cases to be considered and outbreak
perc_strata <- 0.75  # training/test split

### Creating data with no filtering or SMOTE

In [55]:
set.seed(100)
# Choose which dataset to use for testing 
#df <- read_csv("data/merged_data.csv", show_col_types = FALSE) |> clean_names() 
df <- read_csv("data/merged_with_svi.csv", show_col_types = FALSE) |> clean_names()

df$outbreak <- as.integer(df$outbreak >= def_outbreak)

# SVI specific, because it does not include the data for Mcculloch, Mclennan, Mcmullen and Dewitt, 
# if we continue the prev way of data processing it will remove all SVI related features, so we remove these 4 counties.
df <- df[!df$county %in% c("Mcculloch", "Mclennan", "Mcmullen", "Dewitt"), ]

# split the data with stratified sampling
strata <- ifelse(df$outbreak > 0, "nonzero", "zero")
index <- createDataPartition(strata, p = perc_strata, list = FALSE)
train <- df[index,]
test <- df[-index,]
train <- train[, !names(train) %in% c("county")]
test <- test[, !names(test) %in% c("county")]

### Xgboost Model

In [56]:
set.seed(100)
cutoff = 0.5

X_train <- as.matrix(train[, names(train) != "outbreak"])
y_train <- as.numeric(as.character(train$outbreak))

X_test  <- as.matrix(test[, names(test) != "outbreak"])
y_test  <- as.numeric(as.character(test$outbreak))

dtrain  <- xgb.DMatrix(data = X_train, label = y_train)
dtest   <- xgb.DMatrix(data = X_test,  label = y_test)

boost_model <- xgb.train(
  params = list(
    objective   = "binary:logistic",
    eval_metric = "logloss"
  ),
  data    = dtrain,
  nrounds = 500
)

pred_xgb <- as.integer(predict(boost_model, dtest)>= cutoff)

boost_table <- table(Actual = y_test, Predicted = pred_xgb)
boost_table

# importance graph
importance_matrix <- xgb.importance(model = boost_model)
#xgb.plot.importance(importance_matrix)
#importance_matrix[order(importance_matrix$Gain, decreasing = TRUE)]

# Summary of first tree model
# tree_df <- xgb.model.dt.tree(model = boost_model)
# print(tree_df[Tree == 0, ])

TP <- boost_table[2, 2] # True Positives (Actual Outbreak, Predicted Outbreak)
TN <- boost_table[1, 1] # True Negatives (Actual No Outbreak, Predicted No Outbreak)
FP <- boost_table[1, 2] # False Positives (Actual No Outbreak, Predicted Outbreak)
FN <- boost_table[2, 1] # False Negatives (Actual Outbreak, Predicted No Outbreak)

accuracy    <- (TP + TN) / sum(boost_table)
recall <- TP / (TP + FN) 
precision   <- TP / (TP + FP)

cat("Model Performance Summary (Cutoff = 0.5):\n",
    "Accuracy (Overall Correctness): ", round(accuracy, 4), "\n",
    "Recall (Outbreak Capture Rate): ", round(recall, 4), "\n",
    "Precision (Reliability of Outbreak Prediction): ", round(precision, 4), "\n")

      Predicted
Actual  0  1
     0 53  4
     1  3  2

Model Performance Summary (Cutoff = 0.5):
 Accuracy (Overall Correctness):  0.8871 
 Recall (Outbreak Capture Rate):  0.4 
 Precision (Reliability of Outbreak Prediction):  0.3333 


### Parameter Tuning

In [78]:
set.seed(100)
# Xgboost 
c_xgb <- c(seq(0.2, 0.5, by=0.1))      # deciding what counts as outbreak
max_depth = c(seq(3, 9, by=2))         # max depth of each tree 
eta = c(0.01,seq(0.05, 0.4, by=0.05))  # learning rate

# Create grid
param_grid <- expand.grid(
  max_depth = max_depth,
  eta = eta,
  cutoff = c_xgb
)

# Create folds for testing
folds <- createFolds(train$outbreak, k=10)

### Cross validation w/ parameter tuning

In [79]:
set.seed(100)
results_xgb <- data.frame()

for(i in 1:nrow(param_grid)){
  fold_precision <- c()
  fold_recall    <- c()
  fold_accuracy  <- c()
    
  for(fold in folds){
    fold_train <- train[-fold, ]
    fold_test  <- train[fold, ]

    X_fold_train <- as.matrix(fold_train[, names(fold_train) != "outbreak"])
    y_fold_train <- as.numeric(as.character(fold_train$outbreak))
    X_fold_test  <- as.matrix(fold_test[, names(fold_test) != "outbreak"])
    y_fold_test  <- as.numeric(as.character(fold_test$outbreak))

    dtrain_fold <- xgb.DMatrix(data = X_fold_train, label = y_fold_train)
    dtest_fold  <- xgb.DMatrix(data = X_fold_test,  label = y_fold_test)

    model <- xgb.train(
      params = list(
        objective   = "binary:logistic",
        eval_metric = "logloss",
        max_depth   = param_grid$max_depth[i],
        eta         = param_grid$eta[i]
      ),
      data    = dtrain_fold,
      nrounds = 500,
      verbose = 0
    )

    preds <- as.numeric(predict(model, dtest_fold) >= param_grid$cutoff[i])

    # build full 2x2 confusion matrix even if a class is missing
    cm <- table(
      factor(fold_test$outbreak, levels = c(0, 1)),
      factor(preds,              levels = c(0, 1))
    )

    tp <- cm[2, 2]
    fp <- cm[1, 2]
    fn <- cm[2, 1]

    precision <- if((tp + fp) == 0) NA else tp / (tp + fp)
    recall    <- if((tp + fn) == 0) NA else tp / (tp + fn)
    accuracy  <- sum(diag(cm)) / sum(cm)

    fold_precision <- c(fold_precision, precision)
    fold_recall    <- c(fold_recall,    recall)
    fold_accuracy  <- c(fold_accuracy,  accuracy)
  }

  # outside inner loop
  results_xgb <- rbind(results_xgb, data.frame(
    max_depth     = param_grid$max_depth[i],
    eta           = param_grid$eta[i],
    cutoff        = param_grid$cutoff[i],
    precision     = mean(fold_precision, na.rm = TRUE),
    precision_std = sd(fold_precision,   na.rm = TRUE),
    recall        = mean(fold_recall,    na.rm = TRUE),
    recall_std    = sd(fold_recall,      na.rm = TRUE),
    accuracy      = mean(fold_accuracy,  na.rm = TRUE),
    accuracy_std  = sd(fold_accuracy,    na.rm = TRUE)
  ))
}

best_xgb <- results_xgb[which.max(results_xgb$recall), ]
print(best_xgb)

   max_depth  eta cutoff precision precision_std recall recall_std  accuracy
13         3 0.15    0.2  0.462963     0.4697648   0.45   0.475094 0.8894737
   accuracy_std
13   0.08395429


### Evaluating Optimized parameter metrics

In [84]:
set.seed(100)

# Setting optimal parameters
max_depth = 3 
eta = 0.15
cutoff = 0.2 

# Create matricies for xgboost 
X_train <- as.matrix(train[, names(train) != "outbreak"])
y_train <- as.numeric(as.character(train$outbreak))

X_test  <- as.matrix(test[, names(test) != "outbreak"])
y_test  <- as.numeric(as.character(test$outbreak))

dtrain  <- xgb.DMatrix(data = X_train, label = y_train)
dtest   <- xgb.DMatrix(data = X_test,  label = y_test)


# Train model with parameters
final_model <- xgb.train(
  params = list(
    objective   = "binary:logistic",
    eval_metric = "logloss",
    eta = eta, 
    max_depth = max_depth
  ),
  data    = dtrain,
  nrounds = 500
)

# Predict on test set and display results
pred_xgb <- as.integer(predict(final_model, dtest)>= cutoff)
boost_table <- table(Actual = y_test, Predicted = pred_xgb)
boost_table

# importance graph
# Gain: the average imporvement in the models accuracy when the feature is used to split 
# Cover: The average number of observations effected by this feature to split 
# Frequency: The percentage of times a feature is chosen to split the data accross all trees

importance_matrix <- xgb.importance(model = boost_model)
#xgb.plot.importance(importance_matrix)
head(importance_matrix[order(importance_matrix$Gain, decreasing = TRUE)], 10)

# Calculate Precision, Accuracy, and recall
TP <- boost_table[2, 2] # True Positives 
TN <- boost_table[1, 1] # True Negatives 
FP <- boost_table[1, 2] # False Positives 
FN <- boost_table[2, 1] # False Negatives 

accuracy    <- (TP + TN) / sum(boost_table) # total accuracy
recall <- TP / (TP + FN)                    # amount of positives guessed correctly out of all true positives
precision   <- TP / (TP + FP)               # amount of correct positive predictions

# cat("Model Performance Summary:\n",
#     "Accuracy (Overall Correctness): ", round(accuracy, 4), "\n",
#     "Recall (Outbreak Capture Rate): ", round(recall, 4), "\n",
#     "Precision (Reliability of Outbreak Prediction): ", round(precision, 4), "\n")


      Predicted
Actual  0  1
     0 48  9
     1  1  4

Feature,Gain,Cover,Frequency
<chr>,<dbl>,<dbl>,<dbl>
any_cancer_no,0.17733807,0.054234887,0.030
enrollment,0.14269972,0.008588114,0.010
ep_age17,0.11644325,0.077624221,0.040
spl_theme1,0.05678618,0.048553248,0.030
ep_limeng,0.04581185,0.037327218,0.025
m_limeng,0.04542918,0.021467411,0.020
sigm_and_blood_stool_50_75_i_no,0.04494539,0.052558461,0.020
e_munit,0.02868383,0.004250877,0.005
cve,0.02601953,0.072250869,0.080


[1] 1 0 0 0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1
[39] 0 0 0 0 0 0 0 0 1 0 1 1 1 1 0 0 0 0 0 0 0 0 0 1

### Investigating False Results

In [104]:
# Looking at if false negatives is consistent with other classification methods
false_negatives <- test[test$outbreak == 1 & pred_xgb == 0, ]
false_negatives # IS the same county (upshur)


cve,outbreak,enrollment,population,phr,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,⋯,ep_asian,mp_asian,ep_aian,mp_aian,ep_nhpi,mp_nhpi,ep_twomore,mp_twomore,ep_otherrace,mp_otherrace
<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
5,1,7416,44174,4,10.4,7.2,79.3,10.1,13.8,⋯,0.5,0.1,0.1,0.1,0.1,0.1,3.3,0.6,0.2,0.2


### Looking at splits for optimzied tree

In [105]:
# Extract all trees into df 
tree_summary <- xgb.model.dt.tree(model = final_model)

# look at first tree
first_tree <- tree_summary[Tree == 0]
first_tree

false_negatives[, c("any_cancer_no", "enrollment", "m_hh", "ep_aian")]

# SVI documentation: https://www.atsdr.cdc.gov/place-health/media/pdfs/2024/10/SVI2022Documentation.pdf
# any_cancer_no: does not have cancer
# enrollment: percentage of kinds enrolled in school
# m_hh: house holds estimate margin of error? 
# ep_aian: Percentage of American Indian or Alaska Native, not Hispanic or Latino persons estimate, 

# Guide on how to read text dump: 
# look at a given feature (x) then split number (y)
# all formulas will be in the form x < y 
# given this look at the yes or no column the right value is the node that it will traverse to next

# Gain (for non-leaf nodes): "importance level" arbitrary value to determine value 
# Gain (for leaf nodes): probability increase/decrease of outbreak
# Cover: the average number of observations effected by this feature to split 

Tree,Node,ID,Feature,Split,Yes,No,Missing,Gain,Cover
<int>,<int>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>
0,0,0-0,any_cancer_no,94.0,0-1,0-2,0-2,24.15187450,15.462765
0,1,0-1,ep_aian,0.4,0-3,0-4,0-4,2.89536810,11.268079
0,2,0-2,enrollment,1244.0,0-5,0-6,0-6,29.06513210,4.194686
0,3,0-3,Leaf,NA,NA,NA,NA,-0.14863208,9.129611
0,4,0-4,m_hh,308.0,0-7,0-8,0-8,2.14229536,2.138467
0,5,0-5,enrollment,666.0,0-9,0-10,0-10,0.15421069,2.467463
0,6,0-6,Leaf,NA,NA,NA,NA,0.61056948,1.727224
0,7,0-7,Leaf,NA,NA,NA,NA,-0.08521502,1.069234
0,8,0-8,Leaf,NA,NA,NA,NA,0.13225681,1.069234


any_cancer_no,enrollment,m_hh,ep_aian
<dbl>,<dbl>,<dbl>,<dbl>
87.6,7416,294,0.1
